# Stress prediction: preprocessing and hyperparameter tuning
Study notebook for the actual bounded experiment. All outputs are loaded from saved experiment evidence. No training or test evaluation runs automatically.

In [1]:
from pathlib import Path
import json
import pandas as pd
E = Path.cwd()
if not (E/'protocol.json').exists():
    E = E/'experiments/hyperparameter_tuning_20261023'
if not (E/'protocol.json').exists():
    raise RuntimeError('Use the project folder or experiment folder as the working directory.')
def load(name): return json.loads((E/name).read_text())
print('Experiment:', E.name)

Experiment: hyperparameter_tuning_20261023


## 1. Dataset and target
One row represents one respondent. Different respondents can legitimately share all predictor values. Stress frequency is ordered: rarely/never < sometimes < usually/always. Complete-case filtering and unweighted sampling limit generalization.

In [2]:
audit=load('audit.json')
print(json.dumps(audit,indent=2))

{
  "rows": 183040,
  "unique_respondents": 183040,
  "split_counts": {
    "train": 128248,
    "test": 30718,
    "validation": 15197,
    "quarantine": 8877
  },
  "duplicate_full_feature_rows": 118,
  "duplicate_feature_target_rows": 99,
  "profile_groups_crossing_splits": 0,
  "features": [
    "age",
    "sleep_duration",
    "unhealthy_days",
    "bmi",
    "sex",
    "exercise",
    "general_health",
    "marital",
    "education",
    "employment",
    "income",
    "support",
    "isolation",
    "job_loss",
    "food_insecurity",
    "bill_difficulty",
    "utility_threat",
    "transport_barrier",
    "medical_cost"
  ],
  "target_mapping": {
    "4/5": 0,
    "3": 1,
    "1/2": 2
  },
  "train_preprocessor_matches": true,
  "raw_missing_policy": "Complete cases: reject invalid/refused/dont-know codes; physical unhealthy days 88->0; BMI /100.",
  "limitations": [
    "Complete-case US survey; not survey-weighted population estimates.",
    "Predictors and stress are concurr

## 2. Preprocessing
Raw invalid/refused/dont-know codes were excluded; physical unhealthy days 88 becomes 0; BMI is divided by 100. Four numeric features use StandardScaler and 15 categorical features use fixed-category OneHotEncoder, producing 69 features. IDs, raw stress and target are not inputs. Every CV preprocessing object fits only its fitting partition; early-stop, calibration and scoring partitions only transform. The saved original scaler was independently checked against training-only statistics.

In [3]:
protocol=load('protocol.json')
print('Features:', protocol['features'])
print('Original preprocessing matches training:',audit['train_preprocessor_matches'])
roles=pd.read_csv(E/'cv_split_ids.csv')
print(roles.groupby(['fold','role']).size().to_string())
assert not roles.duplicated(['fold','respondent_id']).any()

Features: ['age', 'sleep_duration', 'unhealthy_days', 'bmi', 'sex', 'exercise', 'general_health', 'marital', 'education', 'employment', 'income', 'support', 'isolation', 'job_loss', 'food_insecurity', 'bill_difficulty', 'utility_threat', 'transport_barrier', 'medical_cost']
Original preprocessing matches training: True
fold  role      
0     calibrate      1419
      early_stop     1415
      fit           11329
      score         14162
1     calibrate      1416
      early_stop     1415
      fit           11331
      score         14163


## Reproduce the final-fit preprocessing
This cell fits a fresh demonstration preprocessor on the recorded training-fit respondents only. It checks it against the saved candidate preprocessor. It does not train a model, evaluate the test set, or overwrite any artifacts.

In [4]:
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
schema=load('backup/models/current_models/schema.json')
features=schema['numeric']+schema['categorical']
roles=pd.read_csv(E/'final_train_ids.csv')
fit_ids=set(roles.loc[roles.role=='fit','respondent_id'])
data=pd.read_csv(E/'backup/data/expanded_survey_dataset/respondents.csv')
fit_data=data.loc[data.respondent_id.isin(fit_ids)].copy()
assert fit_data.split.eq('train').all()
assert len(fit_data)==len(fit_ids)
assert not fit_data[features].isna().any().any()
categories=[list(range(schema['limits'][k][0],schema['limits'][k][1]+1)) for k in schema['categorical']]
pre=ColumnTransformer([('numeric',StandardScaler(),schema['numeric']),('categorical',OneHotEncoder(categories=categories,sparse_output=False,handle_unknown='error'),schema['categorical'])])
X=pre.fit_transform(fit_data[features]).astype('float32')
saved=joblib.load(E/'candidates/0/20261023_preprocessor.joblib')
assert np.allclose(X[:100],saved.transform(fit_data[features].iloc[:100]).astype('float32'))
print('Training-fit matrix:',X.shape)
print('Numeric means learned only from fitting respondents:',pre.named_transformers_['numeric'].mean_)
print('Preprocessing matches saved candidate: PASS')
del data,fit_data,X

Training-fit matrix: (115428, 69)
Numeric means learned only from fitting respondents: [55.16841668  7.02910039  4.28312888 28.75154599]
Preprocessing matches saved candidate: PASS


## 3. Tuning without leakage
Training-only outer folds score each configuration. Inner development data choose early-stopping epochs; a separate inner calibration partition chooses the cutoff. Validation is reserved for final thresholds and eligibility. The test was used historically and is explicitly not pristine.

In [5]:
print(json.dumps({k:protocol[k] for k in ['screen','final','selection','promotion','search_scope']},indent=2))

{
  "screen": "Training profiles whose hash integer modulo 4 is zero (~25% training); 2 stratified grouped outer folds; each outer training fold split into 80% fit, 10% early-stop development, 10% threshold calibration using grouped folds. Same fold IDs across trials.",
  "final": "Best training-CV configuration per model; fit 90% of full training split, early stop on remaining grouped 10%; two predetermined seeds for each neural network; first seed is candidate, second is stability evidence only.",
  "selection": "CV rank: minimum recall-constraint violation, then highest mean accuracy, macro-F1 and QWK. Final validation floor=max(.60,original adjusted recall); ensemble floor=max(.60,current ensemble recall). Highest cutoff meeting floor. Candidates must improve accuracy and not reduce macro-F1, QWK or sometimes recall versus matched-threshold original; must also not reduce currently deployed accuracy or high recall. No tolerance except 1e-12 numerical comparisons.",
  "promotion": "M

## 4. Search spaces and selected settings
Three configurations per model, including the original settings. This is a bounded pilot, not an exhaustive search. Existing architecture sizes are fixed; neural learning rate, regularization, batch size, weighting, dropout and scheduler are investigated.

In [6]:
print(json.dumps(load('cv_selected.json'),indent=2))

{
  "Logistic Regression": {
    "config": {
      "id": 1,
      "params": {
        "C": 0.01,
        "class_weight": null,
        "dual": false,
        "fit_intercept": true,
        "intercept_scaling": 1,
        "l1_ratio": null,
        "max_iter": 1200,
        "multi_class": "deprecated",
        "penalty": "l2",
        "random_state": 20261018,
        "solver": "lbfgs",
        "tol": 0.0001,
        "verbose": 0,
        "warm_start": false
      },
      "weight_power": 0.5
    },
    "mean": {
      "accuracy": 0.6742100813193352,
      "macro_f1": 0.5127189024841049,
      "qwk": 0.4871902457119174,
      "high_recall": 0.625598168028669
    },
    "recall_violation": 0
  },
  "Random Forest": {
    "config": {
      "id": 1,
      "params": {
        "bootstrap": true,
        "ccp_alpha": 0.0,
        "class_weight": "balanced",
        "criterion": "gini",
        "max_depth": 12,
        "max_features": 0.6,
        "max_leaf_nodes": null,
        "max_samples": 

## 5. Neural stability
The first seed was predetermined as the candidate. The second seed is reported for stability, not cherry-picked. These are cross-sectional features, not temporal patient measurements.

In [7]:
frozen=load('frozen_selection.json')
for name, item in frozen['models'].items():
    print(name,[(s['seed'],round(s['constrained']['accuracy'],4),len(s['fit']['history']) if isinstance(s['fit']['history'],list) else 'tree') for s in item['seed_reports']])

Logistic Regression [(20261023, 0.674, 0)]
Random Forest [(20261023, 0.6138, 0)]
XGBoost [(20261023, 0.6747, 'tree')]
1D CNN [(20261023, 0.6713, 14), (20261024, 0.6788, 14)]
LSTM [(20261023, 0.6837, 7), (20261024, 0.6816, 7)]
CNN-LSTM [(20261023, 0.6868, 8), (20261024, 0.6788, 8)]


## 6. One frozen final evaluation
Original deployed decisions and matched-threshold original decisions are both shown. Always rarely: 65.78% historical holdout accuracy, 0% high recall. See test_report.json for all precision/recall/F1 values, confusion matrices, paired intervals and AUC. Intervals are descriptive and do not correct for historical test reuse.

In [8]:
table=pd.read_csv(E/'comparison.csv')
print(table.to_string(index=False))
results=load('test_report.json')
print(json.dumps(results['baselines'],indent=2))

              model  old_accuracy  matched_old_accuracy  candidate_accuracy  old_high_recall  matched_old_high_recall  candidate_high_recall  candidate_macro_f1  candidate_qwk  candidate_middle_recall  candidate_auc  validation_eligible  performance_gate_passed
Logistic Regression     67.455564             67.383944           67.445797        64.143646                64.585635              64.861878            0.511413       0.497541                15.933827       0.780395                False                    False
      Random Forest     66.583111             63.353734           61.657660        61.104972                61.187845              62.182320            0.538632       0.491492                42.040342       0.768146                False                    False
            XGBoost     67.543460             67.667166           67.631356        65.331492                65.138122              65.248619            0.524043       0.504964                18.937745       0.78452

In [9]:
for name,item in results.items():
    if name=='baselines': continue
    print(name,'paired intervals:',item['paired_vs_matched'],'performance gate:',item['performance_gate_passed'])

Logistic Regression paired intervals: {'accuracy_delta_95ci': [-0.0001627710137378502, 0.0013672765153981414], 'high_recall_delta_95ci': [0.0005524861878453136, 0.004972375690607711]} performance gate: False
Random Forest paired intervals: {'accuracy_delta_95ci': [-0.019923985936584416, -0.01419363239794258], 'high_recall_delta_95ci': [0.0027624309392264568, 0.0174033149171271]} performance gate: False
XGBoost paired intervals: {'accuracy_delta_95ci': [-0.0023121622501465376, 0.0016285239924474135], 'high_recall_delta_95ci': [-0.0044198895027623974, 0.006353591160220939]} performance gate: False
1D CNN paired intervals: {'accuracy_delta_95ci': [0.007519206979621035, 0.012762875187186697], 'high_recall_delta_95ci': [-0.0038743093922652566, 0.007182320441988965]} performance gate: False
LSTM paired intervals: {'accuracy_delta_95ci': [-0.005404811511166135, -0.0011393970961651734], 'high_recall_delta_95ci': [-0.007182320441988965, 0.0055248618784530246]} performance gate: False
CNN-LSTM p

## 7. Preservation and limitations
The script never installs candidate models. Any accepted candidate still needs inference parity, input validation and memory-limit checks before activation. Failed candidates remain experiments. Exact backups and hashes are preserved. A higher validation recall does not guarantee fewer misses on future respondents.

In [10]:
print('Protected files:',len(load('original_hashes.json')))
print('Frozen selection recorded:',load('TEST_STARTED.json')['frozen_sha256'])
print('Read README.md for interpretation and rollback.')

Protected files: 23
Frozen selection recorded: a3aae252d140fe211ac982c911247955987f35d7cfa3822bf2d18f0ab4f8f958
Read README.md for interpretation and rollback.
